# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima12aa/fa-ml/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
## Finding 1: The Freshness Multiplier (page 9)

**The paper's claim:** Old content (365+ days) that was refreshed within the last 30 days
shows a 3.2x health score boost and 57x more impressions compared to old content that wasn't
recently refreshed.

**My methodology question:** Just two sentences earlier, the paper transparently flags that a
neighboring calculation from the same general population — the 361+ day "growth-to-decline
ratio" of 283:1 — is unreliable, because that bucket contains only 1 declining page. This is
good, honest practice.

However, the "3.2x health boost, 57x impressions" claim draws from a closely related
population (365+ day pages, further narrowed to those specifically refreshed in the last 30
days) — likely an even smaller subgroup than the one already flagged as unstable. The paper
never states the sample size (n) for this specific comparison, so a reader can't tell whether
this headline number rests on a handful of pages (sharing the same fragility as the disclaimed
283:1 ratio) or a genuinely robust sample.

**Constructive suggestion:** Reporting the exact n for the "365+ refreshed vs. not refreshed"
comparison — the same transparency already extended to the 283:1 ratio — would let readers
judge whether this is a robust, actionable finding or a small-sample result that deserves the
same caution.

In [ ]:
## Finding 2: ML Appendix — Feature Importance for Health Score (page 27)

**The paper's claim:** A Random Forest model predicting Health Score found Average Position
(43%), Impressions (32%), and Scroll Depth (15%) as the top features — 90% combined
importance.

**My methodology question:** Health Score is explicitly defined (page 5) as a formula built
directly from Impressions (30 pts), Position (30 pts), CTR (20 pts), and Scroll Depth (20 pts).
Three of the top four "most important" features found by the model — Position, Impressions,
and Scroll Depth — are literal mathematical ingredients of the target itself. This makes the
exercise close to circular: the model isn't discovering a genuine, external pattern about
content performance, it's largely reconstructing a formula it was partially handed the
ingredients to, similar to feeding a model the same value used to compute its own label
(the leakage pattern we specifically learned to avoid in this internship, e.g. excluding
`trend_pct` when predicting `is_declining`).

The paper does disclose this ("importance is descriptive rather than causal") — genuinely
good practice — but doesn't quantify how much of the 90% combined importance is pure formula-
recovery versus real signal, nor does it test the model using only features NOT already part
of the Health Score formula (e.g. Content Age, Word Count, Search Volume), which would show
whether anything genuinely new was learned.

**Constructive suggestion:** Re-running this same feature importance exercise using only the
features that are NOT part of the Health Score formula would show whether the model finds any
independent, non-circular signal — a more informative and less circular version of this
analysis.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
# --- 1. Token + connection ---
import os
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected.")

Connected.


In [2]:
# --- 2. February features (with client_hash_id for grouping) ---
feb_impressions = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) AS feb_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

feb_ctr = con.sql(f"""
    SELECT content_hash_id,
        SUM(gsc_clicks) AS feb_clicks,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
             ELSE NULL END AS feb_ctr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feb_position = con.sql(f"""
    SELECT content_hash_id, AVG(gsc_avg_position) AS feb_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE AND gsc_avg_position > 0
    GROUP BY content_hash_id
""").df()

content_features = con.sql(f"""
    SELECT content_hash_id, word_count, search_volume
    FROM {TABLES['dim_content']}
""").df()

print("feb_impressions:", feb_impressions.shape)
print("feb_ctr:", feb_ctr.shape)
print("feb_position:", feb_position.shape)
print("content_features:", content_features.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feb_impressions: (153559, 3)
feb_ctr: (153559, 3)
feb_position: (151956, 2)
content_features: (519606, 3)


In [3]:
# --- 3. March impressions + label ---
march_impressions = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS march_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

trend_data = feb_impressions.merge(march_impressions, on="content_hash_id", how="inner")
trend_data["is_declining"] = (
    trend_data["march_impressions"] < 0.8 * trend_data["feb_impressions"]
).astype(int)

print("trend_data:", trend_data.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

trend_data: (134238, 5)


In [4]:
# --- 4. Assemble final training table (with client_hash_id) ---
training_table = content_features \
    .merge(feb_impressions, on="content_hash_id", how="inner") \
    .merge(feb_ctr, on="content_hash_id", how="left") \
    .merge(feb_position, on="content_hash_id", how="left") \
    .merge(trend_data[["content_hash_id", "is_declining"]], on="content_hash_id", how="inner")

feature_cols = ["word_count", "search_volume", "feb_impressions", "feb_ctr", "feb_avg_position"]
training_table_clean = training_table.dropna(subset=feature_cols)

print("training_table_clean:", training_table_clean.shape)
print(training_table_clean.columns.tolist())

training_table_clean: (75189, 9)
['content_hash_id', 'word_count', 'search_volume', 'client_hash_id', 'feb_impressions', 'feb_clicks', 'feb_ctr', 'feb_avg_position', 'is_declining']


# ============================================================
# SETUP SUMMARY
# ------------------------------------------------------------
# feb_impressions (153,559 rows): one row per page, total February GSC
#   impressions, plus client_hash_id (needed for grouped splitting).
# feb_ctr (153,559 rows): one row per page, February clicks + computed CTR.
# feb_position (151,956 rows): one row per page, average February position
#   (excludes avg_position=0 sentinel rows, per our w03 finding).
# content_features (519,606 rows): ALL pages in dim_content, with static
#   word_count and search_volume (no time-window filtering needed).
#
# trend_data (134,238 rows): pages with BOTH real Feb and March GSC data,
#   with is_declining computed as (march_impressions < 0.8 * feb_impressions)
#   -- same 20%-drop threshold as w03.
#
# training_table_clean (75,189 rows, 9 columns): final assembled table --
#   all 5 features + client_hash_id + label, with NaN rows dropped.
#   Same size as our w05 training table, confirming a clean, consistent
#   rebuild.
# ============================================================

In [5]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

feature_cols = ["word_count", "search_volume", "feb_impressions", "feb_ctr", "feb_avg_position"]
X = training_table_clean[feature_cols]
y = training_table_clean["is_declining"]
groups = training_table_clean["client_hash_id"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# --- BEFORE: plain ungrouped split (dishonest) ---
X_tr_bad, X_te_bad, y_tr_bad, y_te_bad = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Check: does any client appear in both train and test? (it's allowed to here)
train_idx_bad = X_tr_bad.index
test_idx_bad = X_te_bad.index
train_clients_bad = set(training_table_clean.loc[train_idx_bad, "client_hash_id"])
test_clients_bad = set(training_table_clean.loc[test_idx_bad, "client_hash_id"])
overlap_bad = train_clients_bad & test_clients_bad

model_bad = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_bad, y_tr_bad)
scores_bad = model_bad.predict_proba(X_te_bad)[:, 1]

print("=== BEFORE: ungrouped split ===")
print("Overlapping clients (train AND test):", len(overlap_bad))
print("Precision@20:", precision_at_k(scores_bad, y_te_bad.values, 20))
print("Precision@50:", precision_at_k(scores_bad, y_te_bad.values, 50))

=== BEFORE: ungrouped split ===
Overlapping clients (train AND test): 35
Precision@20: 0.6
Precision@50: 0.52


In [6]:
# --- AFTER: grouped split by client_hash_id (honest, same as w05) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx_good, test_idx_good = next(gss.split(X, y, groups=groups))

X_tr_good, X_te_good = X.iloc[train_idx_good], X.iloc[test_idx_good]
y_tr_good, y_te_good = y.iloc[train_idx_good], y.iloc[test_idx_good]

train_clients_good = set(training_table_clean.iloc[train_idx_good]["client_hash_id"])
test_clients_good = set(training_table_clean.iloc[test_idx_good]["client_hash_id"])
overlap_good = train_clients_good & test_clients_good

model_good = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_good, y_tr_good)
scores_good = model_good.predict_proba(X_te_good)[:, 1]

print("=== AFTER: grouped (client-holdout) split ===")
print("Overlapping clients (train AND test):", len(overlap_good))
print("Precision@20:", precision_at_k(scores_good, y_te_good.values, 20))
print("Precision@50:", precision_at_k(scores_good, y_te_good.values, 50))

=== AFTER: grouped (client-holdout) split ===
Overlapping clients (train AND test): 0
Precision@20: 0.15
Precision@50: 0.22


In [7]:
# --- Side-by-side comparison table ---
comparison = pd.DataFrame({
    "split_type": ["ungrouped (before)", "grouped/client-holdout (after)"],
    "client_overlap": [len(overlap_bad), len(overlap_good)],
    "precision_at_20": [
        precision_at_k(scores_bad, y_te_bad.values, 20),
        precision_at_k(scores_good, y_te_good.values, 20)
    ],
    "precision_at_50": [
        precision_at_k(scores_bad, y_te_bad.values, 50),
        precision_at_k(scores_good, y_te_good.values, 50)
    ]
})
print(comparison)

                       split_type  client_overlap  precision_at_20  \
0              ungrouped (before)              35             0.60   
1  grouped/client-holdout (after)               0             0.15   

   precision_at_50  
0             0.52  
1             0.22  


## Model under an honest split (before/after)

**Before (ungrouped train_test_split):** 35 clients appeared in BOTH train and test.
Precision@20 = 0.60, Precision@50 = 0.52 — inflated, since the model could partly recognize
client-specific patterns from pages of the same client seen during training, rather than
genuinely predicting decline for unseen content.

**After (GroupShuffleSplit, grouped by client_hash_id):** 0 client overlap — verified,
confirmed programmatically. Precision@20 = 0.15, Precision@50 = 0.22 — a large drop, but this
is the honest, trustworthy number, since it reflects performance on genuinely unseen clients.

**Takeaway:** The gap between these two numbers (0.60→0.15 at K=20) is itself the finding —
concrete proof that an ungrouped split can dramatically overstate real-world performance for
this kind of client-based data. This matches the exact concern flagged in our w03 data
contract work and the starter pipeline's own use of client_holdout validation.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.